# Question 1

### Preprocessing - Lowercasing and Stopword Removal

In [ ]:
import pandas as pd
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk


text= ["I really like the phone","Battery life is amazing","Worst mobile ever","I dislike the product","Great performance and speed","Very poor quality product","Camera and performance are excellent","Bad customer service experience"]
labels= ["Positive","Positive","Negative","Negative","Positive","Negative","Positive","Negative"]


try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')
try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab')

# DataFrame
df = pd.DataFrame({'text': text, 'label': labels})

# Preprocessing function
def preprocess_text(text):
    text = text.lower()
    tokens = word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    filtered_tokens = [word for word in tokens if word.isalnum() and word not in stop_words]
    return ' '.join(filtered_tokens)

df['processed_text'] = df['text'].apply(preprocess_text)
display(df.head())

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


,text,label,processed_text
0,I really like the phone,Positive,really like phone
1,Battery life is amazing,Positive,battery life amazing
2,Worst mobile ever,Negative,worst mobile ever
3,I dislike the product,Negative,dislike product
4,Great performance and speed,Positive,great performance speed


### Task 2: Apply TF-IDF Vectorization

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=1000)

X = tfidf_vectorizer.fit_transform(df['processed_text'])
y = df['label']

print("TF-IDF Vectorization complete. Shape of X:", X.shape)

TF-IDF Vectorization complete. Shape of X: (8, 22)


### Task 3: Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

# data split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")

Training set size: 6 samples
Testing set size: 2 samples


### Task 4: Train Sentiment Analysis Models

a. Multinomial Naive Bayes
b. Logistic Regression

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression

# Multinomial Naive Bayes model
mnb_model = MultinomialNB()
mnb_model.fit(X_train, y_train)
print("Multinomial Naive Bayes model trained.")

# Logistic Regression model
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)
print("Logistic Regression model trained.")

Multinomial Naive Bayes model trained.
Logistic Regression model trained.


### Task 5: Predict sentiment for a new phrase

In [ ]:
new_phrase = "This mobile has excellent camera and amazing performance"


processed_new_phrase = preprocess_text(new_phrase)


X_new_phrase = tfidf_vectorizer.transform([processed_new_phrase])

#  sentiment prediction using Multinomial Naive Bayes
mnb_prediction = mnb_model.predict(X_new_phrase)[0]
print(f"Sentiment prediction (Multinomial Naive Bayes) for '{new_phrase}': {mnb_prediction}")

#  sentiment prediction using Logistic Regression
lr_prediction = lr_model.predict(X_new_phrase)[0]
print(f"Sentiment prediction (Logistic Regression) for '{new_phrase}': {lr_prediction}")

Sentiment prediction (Multinomial Naive Bayes) for 'This mobile has excellent camera and amazing performance': Positive
Sentiment prediction (Logistic Regression) for 'This mobile has excellent camera and amazing performance': Positive


### Task 6: Print accuracy of both models and prediction result

In [ ]:
from sklearn.metrics import accuracy_score

# Evaluation Multinomial Naive Bayes model
mnb_y_pred = mnb_model.predict(X_test)
mnb_accuracy = accuracy_score(y_test, mnb_y_pred)
print(f"Multinomial Naive Bayes Model Accuracy: {mnb_accuracy:.4f}")

# Evaluation Logistic Regression model
lr_y_pred = lr_model.predict(X_test)
lr_accuracy = accuracy_score(y_test, lr_y_pred)
print(f"Logistic Regression Model Accuracy: {lr_accuracy:.4f}")

Multinomial Naive Bayes Model Accuracy: 1.0000
Logistic Regression Model Accuracy: 1.0000


# Question 2

### CountVectorizer Matrix

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

documents = [
    "machine learning for NLP tasks",
    "deep learning for NLP tasks",
    "pizza and burgers are tasty",
    "AI improves NLP systems",
    "machine learning improves AI models"
]

vectorizer = CountVectorizer()
X_counts = vectorizer.fit_transform(documents).toarray()

print("Vocabulary:", vectorizer.get_feature_names_out())
print("Count Matrix:\n", X_counts)

Vocabulary: ['ai' 'and' 'are' 'burgers' 'deep' 'for' 'improves' 'learning' 'machine'
 'models' 'nlp' 'pizza' 'systems' 'tasks' 'tasty']
Count Matrix:
 [[0 0 0 0 0 1 0 1 1 0 1 0 0 1 0]
 [0 0 0 0 1 1 0 1 0 0 1 0 0 1 0]
 [0 1 1 1 0 0 0 0 0 0 0 1 0 0 1]
 [1 0 0 0 0 0 1 0 0 0 1 0 1 0 0]
 [1 0 0 0 0 0 1 1 1 1 0 0 0 0 0]]


### Cosine Similarity and Matrix Building

In [ ]:
def cosine_similarity_manual(vec1, vec2):
    # a. Dot product
    dot_product = np.dot(vec1, vec2)

    # b. Magnitude calculation
    mag1 = np.sqrt(np.sum(vec1**2))
    mag2 = np.sqrt(np.sum(vec2**2))

    if mag1 == 0 or mag2 == 0:
        return 0.0
    return dot_product / (mag1 * mag2)

# full similarity matrix
num_docs = len(documents)
sim_matrix = np.zeros((num_docs, num_docs))

for i in range(num_docs):
    for j in range(num_docs):
        sim_matrix[i][j] = cosine_similarity_manual(X_counts[i], X_counts[j])

# DataFrame
sim_df = pd.DataFrame(sim_matrix, columns=[f"Doc {i}" for i in range(num_docs)], index=[f"Doc {i}" for i in range(num_docs)])
display(sim_df)

,Doc 0,Doc 1,Doc 2,Doc 3,Doc 4
Doc 0,1.000000,0.800000,0.0,0.223607,0.400000
Doc 1,0.800000,1.000000,0.0,0.223607,0.200000
Doc 2,0.000000,0.000000,1.0,0.000000,0.000000
Doc 3,0.223607,0.223607,0.0,1.000000,0.447214
Doc 4,0.400000,0.200000,0.0,0.447214,1.000000


### Most and Least Similar Document Pairs

In [ ]:
most_similar = -1
least_similar = 2
pair_most = None
pair_least = None

for i in range(num_docs):
    for j in range(i + 1, num_docs):
        sim = sim_matrix[i][j]

        if sim > most_similar:
            most_similar = sim
            pair_most = (i, j)

        if sim < least_similar:
            least_similar = sim
            pair_least = (i, j)

print(f"Most Similar: Document {pair_most[0]} and Document {pair_most[1]} (Score: {most_similar:.4f})")
print(f"Doc {pair_most[0]}: {documents[pair_most[0]]}")
print(f"Doc {pair_most[1]}: {documents[pair_most[1]]}")

print(f"\nLeast Similar: Document {pair_least[0]} and Document {pair_least[1]} (Score: {least_similar:.4f})")
print(f"Doc {pair_least[0]}: {documents[pair_least[0]]}")
print(f"Doc {pair_least[1]}: {documents[pair_least[1]]}")

Most Similar: Document 0 and Document 1 (Score: 0.8000)
Doc 0: machine learning for NLP tasks
Doc 1: deep learning for NLP tasks

Least Similar: Document 0 and Document 2 (Score: 0.0000)
Doc 0: machine learning for NLP tasks
Doc 2: pizza and burgers are tasty


#Question 3

### Vocabulary Building and Vector Creation

In [ ]:
import numpy as np
import pandas as pd

documents_q3 = [
    "I like studying NLP concepts",
    "NLP is very powerful and useful",
    "I enjoy AI with NLP applications",
    "AI and NLP are changing the world"
]

# 1. Vocabulary builiding
def get_vocab(docs):
    vocab = set()
    for doc in docs:
        words = doc.lower().split()
        vocab.update(words)
    return sorted(list(vocab))

vocab = get_vocab(documents_q3)
word_to_idx = {word: i for i, word in enumerate(vocab)}
print(f"Vocabulary: {vocab}")

# 2a. Binary vectors
def get_binary_vectors(docs, vocab):
    vectors = []
    for doc in docs:
        words = set(doc.lower().split())
        vec = [1 if word in words else 0 for word in vocab]
        vectors.append(vec)
    return np.array(vectors)

# 2b. Frequency vectors
def get_frequency_vectors(docs, vocab):
    vectors = []
    for doc in docs:
        words = doc.lower().split()
        vec = [words.count(word) for word in vocab]
        vectors.append(vec)
    return np.array(vectors)

binary_vecs = get_binary_vectors(documents_q3, vocab)
freq_vecs = get_frequency_vectors(documents_q3, vocab)

print("\nBinary Vectors:\n", binary_vecs)
print("\nFrequency Vectors:\n", freq_vecs)

Vocabulary: ['ai', 'and', 'applications', 'are', 'changing', 'concepts', 'enjoy', 'i', 'is', 'like', 'nlp', 'powerful', 'studying', 'the', 'useful', 'very', 'with', 'world']

Binary Vectors:
 [[0 0 0 0 0 1 0 1 0 1 1 0 1 0 0 0 0 0]
 [0 1 0 0 0 0 0 0 1 0 1 1 0 0 1 1 0 0]
 [1 0 1 0 0 0 1 1 0 0 1 0 0 0 0 0 1 0]
 [1 1 0 1 1 0 0 0 0 0 1 0 0 1 0 0 0 1]]

Frequency Vectors:
 [[0 0 0 0 0 1 0 1 0 1 1 0 1 0 0 0 0 0]
 [0 1 0 0 0 0 0 0 1 0 1 1 0 0 1 1 0 0]
 [1 0 1 0 0 0 1 1 0 0 1 0 0 0 0 0 1 0]
 [1 1 0 1 1 0 0 0 0 0 1 0 0 1 0 0 0 1]]


### Euclidean Distance and Similarity Matrix

In [ ]:
# 3. euclidean distance
def euclidean_distance(vec1, vec2):
    return np.sqrt(np.sum((vec1 - vec2)**2))

# 4. full pair-wise distance matrix
num_docs_q3 = len(documents_q3)
dist_matrix = np.zeros((num_docs_q3, num_docs_q3))

for i in range(num_docs_q3):
    for j in range(num_docs_q3):
        dist_matrix[i][j] = euclidean_distance(freq_vecs[i], freq_vecs[j])

dist_df = pd.DataFrame(dist_matrix, columns=[f"Doc {i}" for i in range(num_docs_q3)], index=[f"Doc {i}" for i in range(num_docs_q3)])
display(dist_df)

,Doc 0,Doc 1,Doc 2,Doc 3
Doc 0,0.000000,3.000000,2.645751,3.162278
Doc 1,3.000000,0.000000,3.162278,3.000000
Doc 2,2.645751,3.162278,0.000000,3.000000
Doc 3,3.162278,3.000000,3.000000,0.000000


### Most and Least Similar Pairs

In [ ]:
# 5. most/least similar
min_dist = float('inf')
max_dist = -1.0
pair_most_sim = None
pair_least_sim = None

for i in range(num_docs_q3):
    for j in range(i + 1, num_docs_q3):
        d = dist_matrix[i][j]
        if d < min_dist:
            min_dist = d
            pair_most_sim = (i, j)
        if d > max_dist:
            max_dist = d
            pair_least_sim = (i, j)

print(f"Most Similar (Min Distance): Doc {pair_most_sim[0]} and Doc {pair_most_sim[1]} (Distance: {min_dist:.4f})")
print(f"Least Similar (Max Distance): Doc {pair_least_sim[0]} and Doc {pair_least_sim[1]} (Distance: {max_dist:.4f})")

Most Similar (Min Distance): Doc 0 and Doc 2 (Distance: 2.6458)
Least Similar (Max Distance): Doc 0 and Doc 3 (Distance: 3.1623)


#Question 4

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import Counter
import math

# Input documents
docs = [
    "Deep learning is powerful",
    "Deep learning is useful",
    "Machine learning is powerful",
    "Deep learning is powerful and useful"
]

# 1. Manual Computation
def manual_tfidf(docs):
    # Tokenize
    tokenized_docs = [d.lower().split() for d in docs]
    vocab = sorted(list(set(word for d in tokenized_docs for word in d)))
    num_docs = len(docs)

    # DF
    df_counts = {word: sum(1 for d in tokenized_docs if word in d) for word in vocab}

    # IDF
    idf = {word: math.log(num_docs / count) for word, count in df_counts.items()}

    # TF and TF-IDF matrix
    tfidf_matrix = []
    for d in tokenized_docs:
        counts = Counter(d)
        doc_tfidf = []
        for word in vocab:
            tf = counts[word] / len(d)
            doc_tfidf.append(tf * idf[word])
        tfidf_matrix.append(doc_tfidf)

    return vocab, np.array(tfidf_matrix), idf

vocab_manual, matrix_manual, idf_manual = manual_tfidf(docs_q4)

# 2. Sklearn TF-IDF
vectorizer = TfidfVectorizer(token_pattern=r"\b\w+\b")
sklearn_matrix = vectorizer.fit_transform(docs_q4).toarray()
vocab_sklearn = vectorizer.get_feature_names_out()

# 3. Print Results
print(" Manual Vocabulary ")
print(vocab_manual)

print("\n Manual TF-IDF Matrix ")
df_manual = pd.DataFrame(matrix_manual, columns=vocab_manual)
display(df_manual)

print("\n Sklearn TF-IDF Matrix ")
df_sklearn = pd.DataFrame(sklearn_matrix, columns=vocab_sklearn)
display(df_sklearn)

# 4. Compare
print("\n Comparison Analysis ")
print(f"Vocabulary Match: {list(vocab_manual) == list(vocab_sklearn)}")
print("Note: Differences in values arise because sklearn uses smoothing (l2 normalization and adding 1 to IDF) by default.")

 Manual Vocabulary 
['and', 'deep', 'is', 'learning', 'machine', 'powerful', 'useful']

 Manual TF-IDF Matrix 


,and,deep,is,learning,machine,powerful,useful
0,0.000000,0.071921,0.0,0.0,0.000000,0.071921,0.000000
1,0.000000,0.071921,0.0,0.0,0.000000,0.000000,0.173287
2,0.000000,0.000000,0.0,0.0,0.346574,0.071921,0.000000
3,0.231049,0.047947,0.0,0.0,0.000000,0.047947,0.115525



 Sklearn TF-IDF Matrix 


,and,deep,is,learning,machine,powerful,useful
0,0.000000,0.547436,0.447565,0.447565,0.000000,0.547436,0.000000
1,0.000000,0.508819,0.415993,0.415993,0.000000,0.000000,0.628493
2,0.000000,0.000000,0.373502,0.373502,0.715739,0.456847,0.000000
3,0.579182,0.369685,0.302241,0.302241,0.000000,0.369685,0.456634



 Comparison Analysis 
Vocabulary Match: True
Note: Differences in values arise because sklearn uses smoothing (l2 normalization and adding 1 to IDF) by default.
